# members

> Adapters that run the three existing agents through `runTeam`.

Each `event` is the body of the agent's old training loop, in the same order and with the same random draws; the old per-event row becomes `extras`, and the old time update becomes `dt`.

In [ ]:
#| default_exp members

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import math

from explRL.circuit import goalStats
from explRL.multi import Member, Outcome
from explRL.prestructured import wrap

In [ ]:
#| export
class FlexibleMember(Member):
    "One 1 ms fixation sample or one saccade of a `flexibleAgent` in a `HeadingArena` (time in samples)."
    timeUnit = 'sample'
    extraNames = ('theta', 'saccade', 'dt', 'reward')

    def __init__(self, agent):
        self.agent = agent

    def event(self, world, t):
        ag = self.agent
        step = ag.rl.chooseAction(ag)
        new_theta, r = world.step(ag.theta, step.dx)
        ag.rl.updateParams(ag, step, r)
        ag.theta = new_theta
        return Outcome(step.dt, 'saccade' if step.saccade else 'fixation', new_theta, r,
                       (step.theta, step.saccade, step.dt, r))

In [ ]:
#| export
class PrestructuredMember(Member):
    "One fixation, saccade and possible jump of a `prestructuredAgent` in a `HeadingArena` (seconds)."
    timeUnit = 's'
    extraNames = ('thetaA', 'thetaC', 'fixDur', 'direction', 'jumped', 'reward', 'thetaG')

    def __init__(self, agent):
        self.agent = agent

    def event(self, world, t):
        ag, p = self.agent, self.agent.params
        ev = ag.rl.chooseAction(ag)
        thetaA1, r = world.step(ag.theta, ev.dx)
        ag.rl.updateParams(ag, ev, r)
        row = (ev.thetaA, ev.thetaC, ev.fixDur, ev.direction, ev.jumped, r, ag.policy.thetaG)
        ag.theta, ag.thetaC = wrap(thetaA1), ev.thetaC_next
        return Outcome(ev.fixDur + p.tS, 'event', ag.theta, r, row)

In [ ]:
#| export
class CircuitMember(Member):
    "One saccade (learning at each grid step), possible jump and fixation of a `circuitAgent` in a `ZoneArena`."
    timeUnit = 's'
    extraNames = ('a', 'c', 'direction', 'steps', 'jumped', 'valence', 'fixDur', 'thetaG', 'strength', 'mirror')

    def __init__(self, agent):
        self.agent = agent

    def event(self, world, t):
        ag, rl, p = self.agent, self.agent.rl, self.agent.params
        sac = rl.chooseAction(ag)
        for _ in range(sac.steps):
            if rl.learnC:
                rl.updateCompass(ag, world.ringActivity(ag.a), sac.size)
            ag.move(sac.direction)
        jumped = rl.jumps and ag.maybeJump(world.ringActivity(ag.a))
        fix = ag.sampleFixation()
        R = self.reward(world, t, fix)
        if rl.learnG and R != 0:
            rl.updateGoal(ag, R, math.ceil(fix / p.dtG))
        gs = goalStats(ag.policy.wG)
        row = (ag.a, ag.c, sac.direction, sac.steps, jumped, R, fix,
               gs['thetaG'], gs['strength'], (ag.a + ag.c) % p.nCirc)
        return Outcome(p.tS + fix, 'iteration', ag.a, float(R), row)

    def reward(self, world, t, fix):
        return world.valence(self.agent.a)

## Tests

Exactness against the old loops is tested through the wrappers with the golden fixtures (AC-1, in `train`, `prestructured_train` and `circuit_train`). Here: declared units and names, the import rule (NFR-5), and independence of agents that share a world (AC-2).

In [ ]:
import ast, inspect
from math import pi
from fastcore.test import test_eq, test_fail
import explRL.members as M
from explRL.core import loadAgent, loadCircuitAgent, loadPrestructuredAgent, loadEnv
from explRL.envs import ZoneArena
from explRL.multi import runTeam

test_eq((FlexibleMember.timeUnit, PrestructuredMember.timeUnit, CircuitMember.timeUnit), ('sample', 's', 's'))
test_eq((len(FlexibleMember.extraNames), len(PrestructuredMember.extraNames), len(CircuitMember.extraNames)), (4, 7, 10))
mods = {n.module for n in ast.walk(ast.parse(inspect.getsource(M))) if isinstance(n, ast.ImportFrom)}
assert not any('train' in m for m in mods), mods
test_fail(lambda: runTeam([FlexibleMember(loadAgent(seed=0)), CircuitMember(loadCircuitAgent(seed=0))], None, 1.0),
          contains='time unit')

In [ ]:
def sameAsSolo(team, solo, i):
    a, b = team.forAgent(i), solo.records
    test_eq(len(a), len(b))
    for x, y in zip(a, b):
        test_eq(x._replace(agent=0), y)

zone = ZoneArena(symmetric=True)
team = runTeam([CircuitMember(loadCircuitAgent(seed=i)) for i in range(3)], zone, 100.0, t0=0.0)
for i in range(3):
    sameAsSolo(team, runTeam([CircuitMember(loadCircuitAgent(seed=i))], zone, 100.0, t0=0.0), i)
ts = [r.t for r in team.records]
test_eq(ts, sorted(ts))

arena = loadEnv(goal=pi)
team = runTeam([PrestructuredMember(loadPrestructuredAgent(seed=i)) for i in range(3)], arena, 240.0, t0=0.0)
for i in range(3):
    sameAsSolo(team, runTeam([PrestructuredMember(loadPrestructuredAgent(seed=i))], arena, 240.0, t0=0.0), i)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()